# Classification Models — Advanced
**Prerequisites:** Complete **Classification Models — Basic** first.

This notebook covers four powerful classification algorithms, then shows how to run all models
in a unified pipeline and tune the best one with Grid Search and Random Search.

## Setup

Condensed setup from the Basic notebook.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.base import clone
from sklearn.preprocessing import FunctionTransformer
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, ConfusionMatrixDisplay, classification_report)

import xgboost as xgb

In [ ]:
df = pd.read_csv('../data/adult.csv', na_values='?')
df.dropna(inplace=True)

edu_dict = {
    'Preschool': 'Elementary', '1st-4th': 'Elementary', '5th-6th': 'Elementary',
    '7th-8th': 'Middle_School', '9th': 'Middle_School', '10th': 'Middle_School', '11th': 'Middle_School',
    '12th': 'High_School', 'HS-grad': 'High_School',
    'Some-college': 'Undergraduate', 'Bachelors': 'Undergraduate',
    'Assoc-voc': 'Undergraduate', 'Prof-school': 'Undergraduate',
    'Masters': 'Graduate', 'Assoc-acdm': 'Graduate', 'Doctorate': 'Graduate',
}
df['education'] = df['education'].replace(edu_dict)

feature_cols = ['age', 'workclass', 'education', 'educational-num', 'marital-status',
                'occupation', 'relationship', 'race', 'gender',
                'capital-gain', 'capital-loss', 'hours-per-week', 'native-country']
X = df[feature_cols]
y = (df['income'] == '>50K').astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric_features     = ['age', 'educational-num', 'capital-gain', 'capital-loss', 'hours-per-week']
categorical_features = ['workclass', 'education', 'marital-status', 'occupation',
                        'relationship', 'race', 'gender', 'native-country']

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
])

def evaluate_clf(name, y_true, y_pred, show_matrix=False):
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    f1   = f1_score(y_true, y_pred, zero_division=0)
    print(f"\n{'─'*45}  {name}")
    print(f"  Accuracy:  {acc:.4f}  |  Precision: {prec:.4f}")
    print(f"  Recall:    {rec:.4f}  |  F1 Score:  {f1:.4f}")
    if show_matrix:
        fig, ax = plt.subplots(figsize=(4, 3))
        ConfusionMatrixDisplay.from_predictions(
            y_true, y_pred, display_labels=['≤50K', '>50K'], cmap='Blues', ax=ax)
        ax.set_title(f'Confusion Matrix — {name}')
        plt.tight_layout()
        plt.show()
    return {'Model': name, 'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1}

print("Setup complete.")

## Naive Bayes

Naive Bayes applies **Bayes' theorem** with the "naive" assumption that all features are
conditionally independent given the class:

$$P(y \mid x_1, x_2, \ldots, x_n) \propto P(y) \prod_{i=1}^{n} P(x_i \mid y)$$

**Why "naive"?** Features like occupation and education are obviously correlated in reality.
Despite this violated assumption, Naive Bayes often works surprisingly well, especially for text.

`GaussianNB` assumes each feature follows a normal distribution within each class.

**Strengths:** Very fast, works well with small datasets, handles high-dimensional data.  
**Weaknesses:** Conditional independence assumption is rarely true; tends to underfit complex patterns.

In [ ]:
from sklearn.naive_bayes import GaussianNB

# GaussianNB requires dense input — ColumnTransformer with OHE produces sparse matrices
to_dense = FunctionTransformer(lambda x: x.toarray(), accept_sparse=True)

clf_nb = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('to_dense', to_dense),          # Convert sparse → dense for GaussianNB
    ('classifier', GaussianNB())
])
clf_nb.fit(X_train, y_train)
y_pred_nb = clf_nb.predict(X_test)

results = [evaluate_clf("Naive Bayes", y_test, y_pred_nb, show_matrix=True)]

print("\nNote: High recall, lower precision — Naive Bayes over-predicts the positive class here.")

## Support Vector Machine (SVM)

SVM finds the **hyperplane that maximises the margin** between the two classes.
The samples closest to the boundary are called **support vectors** — only these influence the boundary.

$$\min \frac{1}{2}\|\mathbf{w}\|^2 \quad \text{subject to} \quad y_i(\mathbf{w} \cdot x_i + b) \geq 1$$

For non-linearly separable data, the **kernel trick** maps features into a higher-dimensional space
where a linear separator exists — without explicitly computing the transformation.

Key parameters:
- **C:** Regularization strength — higher C = smaller margin, fewer misclassifications on training data
- **kernel:** `linear`, `rbf` (radial basis function), `poly`

**Strengths:** Effective in high-dimensional spaces; memory-efficient (only support vectors stored).  
**Weaknesses:** Slow on large datasets; sensitive to feature scale → always scale.

In [ ]:
from sklearn.svm import SVC

# SVC with linear kernel is faster and often comparable to RBF on tabular data
clf_svm = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('to_dense', to_dense),
    ('classifier', SVC(kernel='linear', C=1.0, random_state=42, probability=True))
])
clf_svm.fit(X_train, y_train)
y_pred_svm = clf_svm.predict(X_test)

results.append(evaluate_clf("SVM (linear kernel)", y_test, y_pred_svm, show_matrix=True))

## Random Forest

Random Forest is an **ensemble of Decision Trees** trained with **bagging**:
- Each tree is trained on a bootstrap sample (random subset of data with replacement)
- Each split considers only a random subset of features (`max_features`)
- Prediction = majority vote across all trees

The two sources of randomness make trees diverse — their errors partially cancel out,
resulting in much lower variance than a single tree.

**OOB Score:** Each tree is trained on ~63% of data. The remaining ~37% (out-of-bag)
provides a free validation estimate without a separate validation set.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

clf_rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=100, random_state=42, n_jobs=-1, oob_score=False
    ))
])
clf_rf.fit(X_train, y_train)
y_pred_rf = clf_rf.predict(X_test)

results.append(evaluate_clf("Random Forest (100 trees)", y_test, y_pred_rf, show_matrix=True))

In [ ]:
# Feature importances (averaged across all trees)
rf_model = clf_rf.named_steps['classifier']
ohe_names = clf_rf.named_steps['preprocessor'].transformers_[1][1].get_feature_names_out(categorical_features)
all_names = numeric_features + list(ohe_names)

importances = pd.Series(rf_model.feature_importances_, index=all_names)
top15 = importances.nlargest(15).sort_values()

plt.figure(figsize=(8, 6))
top15.plot(kind='barh', color='steelblue')
plt.title('Random Forest — Top 15 Feature Importances')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## XGBoost

XGBoost (Extreme Gradient Boosting) is a highly optimised implementation of **gradient boosting**.
It builds trees sequentially, where each tree corrects the errors of the previous ones:

$$F_m(x) = F_{m-1}(x) + \eta \cdot h_m(x)$$

Where $\eta$ is the learning rate and $h_m$ is the new tree fitted to the residuals.

XGBoost adds several improvements over classic gradient boosting:
- **Regularization** (L1 and L2 on leaf weights) to prevent overfitting
- **Column subsampling** (like Random Forest) for diversity
- **Efficient handling of missing values** natively
- **Parallel tree building** for speed

It's often the algorithm of choice for tabular data competitions.

In [ ]:
clf_xgb = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', xgb.XGBClassifier(eval_metric='logloss', random_state=42, n_jobs=-1))
])
clf_xgb.fit(X_train, y_train)
y_pred_xgb = clf_xgb.predict(X_test)

results.append(evaluate_clf("XGBoost", y_test, y_pred_xgb, show_matrix=True))

## Full Pipeline Comparison

Now let's compare all 7 classifiers (including the 3 basic ones) in a single run.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

# Full preprocessing pipeline (with imputation for robustness)
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])
full_preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features),
])

all_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(max_depth=5, random_state=42),
    'KNN':                 KNeighborsClassifier(n_neighbors=11),
    'Naive Bayes':         GaussianNB(),
    'SVM':                 SVC(kernel='linear', random_state=42),
    'Random Forest':       RandomForestClassifier(random_state=42, n_jobs=-1),
    'XGBoost':             xgb.XGBClassifier(eval_metric='logloss', random_state=42),
}

summary = []
for name, model in all_models.items():
    # Naive Bayes and SVM need dense input
    if name in ['Naive Bayes', 'SVM']:
        pipe = Pipeline([('pre', full_preprocessor), ('dense', to_dense), ('clf', clone(model))])
    else:
        pipe = Pipeline([('pre', full_preprocessor), ('clf', clone(model))])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    summary.append({
        'Model': name,
        'Accuracy':  round(accuracy_score(y_test, pred), 4),
        'Precision': round(precision_score(y_test, pred, zero_division=0), 4),
        'Recall':    round(recall_score(y_test, pred, zero_division=0), 4),
        'F1':        round(f1_score(y_test, pred, zero_division=0), 4),
    })
    print(f"  ✓ {name}")

In [ ]:
summary_df = pd.DataFrame(summary).set_index('Model').sort_values('F1', ascending=False)
print("\n", summary_df)

In [ ]:
summary_df.plot(kind='bar', figsize=(14, 6), rot=20, width=0.8)
plt.title('All Classification Models — Comparison')
plt.ylabel('Score')
plt.ylim(0, 1)
plt.legend(loc='lower right')
plt.grid(axis='y')
plt.tight_layout()
plt.show()

## Confusion Matrix: All Models at a Glance

Let's visualise where each model makes mistakes.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for ax, (name, model) in zip(axes, all_models.items()):
    if name in ['Naive Bayes', 'SVM']:
        pipe = Pipeline([('pre', full_preprocessor), ('dense', to_dense), ('clf', clone(model))])
    else:
        pipe = Pipeline([('pre', full_preprocessor), ('clf', clone(model))])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    ConfusionMatrixDisplay.from_predictions(
        y_test, pred, display_labels=['≤50K', '>50K'], cmap='Blues', ax=ax, colorbar=False)
    f1 = f1_score(y_test, pred, zero_division=0)
    ax.set_title(f'{name}\nF1={f1:.3f}', fontsize=9)

axes[-1].axis('off')  # Hide unused subplot
plt.suptitle('Confusion Matrices — All Models', fontsize=13)
plt.tight_layout()
plt.show()

## Hyperparameter Tuning with Grid Search

We'll tune XGBoost — the top performer — using `GridSearchCV` with 5-fold cross-validation.

> **Tip:** When tuning with a pipeline, prefix parameter names with the step name and `__`
> (e.g., `model__n_estimators` refers to the `n_estimators` of the `model` step).

In [ ]:
# Rebuild a clean XGBoost pipeline for tuning
xgb_pipeline = Pipeline(steps=[
    ('preprocessor', full_preprocessor),
    ('model', xgb.XGBClassifier(eval_metric='logloss', random_state=42))
])

param_grid = {
    'model__n_estimators':   [100, 200],
    'model__learning_rate':  [0.05, 0.1],
    'model__max_depth':      [3, 5],
    'model__subsample':      [0.8, 1.0],
}

grid_search = GridSearchCV(
    estimator=xgb_pipeline,
    param_grid=param_grid,
    scoring='f1',
    cv=5,
    n_jobs=-1,
    verbose=1,
)

t0 = time.time()
grid_search.fit(X_train, y_train)
print(f"\nGrid Search completed in {time.time()-t0:.1f}s")
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV F1: {grid_search.best_score_:.4f}")

evaluate_clf("XGBoost (Grid Search tuned)", y_test, grid_search.predict(X_test), show_matrix=True)

## Hyperparameter Tuning with Random Search

Random Search is faster when the grid is large. Here we use `scipy.stats` distributions for continuous parameters.

In [ ]:
from scipy.stats import randint, uniform

param_dist = {
    'model__n_estimators':  randint(50, 400),
    'model__learning_rate': uniform(0.01, 0.3),
    'model__max_depth':     randint(2, 8),
    'model__subsample':     uniform(0.6, 0.4),
    'model__colsample_bytree': uniform(0.5, 0.5),
}

random_search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=param_dist,
    n_iter=30,
    scoring='f1',
    cv=5,
    n_jobs=-1,
    random_state=42,
    verbose=1,
)

t0 = time.time()
random_search.fit(X_train, y_train)
print(f"\nRandom Search completed in {time.time()-t0:.1f}s")
print(f"Best parameters: {random_search.best_params_}")
print(f"Best CV F1: {random_search.best_score_:.4f}")

evaluate_clf("XGBoost (Random Search tuned)", y_test, random_search.predict(X_test), show_matrix=True)

In [ ]:
# Final comparison: tuned vs untuned
base_f1  = f1_score(y_test, clf_xgb.predict(X_test))
gs_f1    = f1_score(y_test, grid_search.predict(X_test))
rs_f1    = f1_score(y_test, random_search.predict(X_test))

print("Impact of hyperparameter tuning on F1 Score:")
print(f"  XGBoost (default):      {base_f1:.4f}")
print(f"  XGBoost (Grid Search):  {gs_f1:.4f}  ({(gs_f1-base_f1)*100:+.2f}pp)")
print(f"  XGBoost (Random Search):{rs_f1:.4f}  ({(rs_f1-base_f1)*100:+.2f}pp)")

## Summary

| Model | Key Strengths | Computationally | Best When |
|-------|---------------|-----------------|-----------|
| Naive Bayes | Fast, high recall | Very fast | Small data, text classification |
| SVM | Works well in high dims | Slow on large data | High-dimensional, clear margin |
| Random Forest | Robust, interpretable via importances | Moderate | General purpose, noisy data |
| XGBoost | Top accuracy, regularized | Moderate | Maximum accuracy on tabular data |

**General workflow:**
1. Start with Logistic Regression as a fast baseline
2. Try Random Forest — robust and requires little tuning
3. Try XGBoost — tune with Random Search, then refine with Grid Search on promising ranges
4. Use cross-validation scores to compare, never test-set scores for model selection